# Lab 11: Attention


### Attention is a mechanism that allows a neural network to focus more on relevant information when making a prediction.

Encoder-decoder architecture with attention:
<img src="https://drive.google.com/thumbnail?id=1-QeHdZIeKTo_nAqfh7pRA_o4Nk8CAmP6&sz=w1000" width="700">

## Step 1 — Import libraries

In [8]:
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from google.colab import files

torch.manual_seed(42)

ModuleNotFoundError: No module named 'google.colab'

## Step 2 — Upload the dataset

In [ ]:
df = pd.read_csv("/content/Lab_11_QnA_Dataset_Download.csv")
print("Number of questions:", len(df))
print("Number of answer groups:", df["answer_id"].nunique())
display(df.head())


## Step 3 — Preprocessing

1. Lowercase each question and remove punctuation.
2. Give each word a number.
3. Make all questions the same length by adding zeros (**padding**).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Lowercase
df["question"] = df["question"].str.lower()

# 2. Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["question"])

# Convert words to numbers and add padding
X = pad_sequences(tokenizer.texts_to_sequences(df["question"]), padding="post")

# Answers
y = df["answer_id"].values

# Convert to PyTorch tensors
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

## Step 4 — Build an RNN with attention

- **Embedding** turns word numbers into vectors.
- **RNN** reads the words in sequence.
- **Attention** gives each word a weight.
- **Output** chooses one answer from the dataset.

In [ ]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(tokenizer.word_index) + 1, 16)
        self.rnn = nn.RNN(16, 32)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, df["answer_id"].nunique())

    def forward(self, x):
        x, _ = self.rnn(self.emb(x))
        weights = torch.softmax(self.att(x), dim=1)
        return self.fc((x * weights).sum(1)), weights.squeeze(-1)

model = RNN()

## Step 5 — Train the model
The model learns which answer belongs to each question.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    loss = nn.functional.cross_entropy(model(X)[0], y) #calculate loss to reduce it
    loss.backward() #backpropagation to update parameters
    optimizer.step() #update parameters

print("Training finished!")

## Step 6 — See attention
The bar chart shows how much attention the model gives to each question word.

In [3]:
question = df["question"][0]
words = question.split()

with torch.no_grad():
    prediction, weights = model(X[:1])

plt.bar(words, weights[0, :len(words)].tolist())
plt.title("Attention Weights")
plt.show()

NameError: name 'df' is not defined

## Tasks:


Task 1 — Understand Attention Weights
Easy

Modify Step 6 to visualize the attention weights for three different questions from the dataset instead of only the first question.

Instructions:
- Select three questions from the dataset.
- Generate an attention bar chart for each question.
- Identify the word receiving the highest attention weight in each question.

Question: Do the words with the highest attention weights appear important for understanding the question? Explain your observations.

In [4]:
import matplotlib.pyplot as plt
import torch

indices = [0, 1, 2]

plt.figure(figsize=(15, 4))
for i, idx in enumerate(indices):
    question = df["question"][idx]
    words = question.split()
    
    with torch.no_grad():
        prediction, weights = model(X[idx:idx+1])
    
    attn_weights = weights[0, :len(words)].tolist()
    
    plt.subplot(1, 3, i+1)
    plt.bar(words, attn_weights, color='skyblue')
    plt.title(f"Q{idx}: {question}")
    plt.xticks(rotation=45)
    
    max_weight = max(attn_weights)
    max_word = words[attn_weights.index(max_weight)]
    print(f"Question {idx} ('{question}') -> Highest attention word: '{max_word}'")

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

<Figure size 1500x400 with 0 Axes>

Task 2 — Compare Attention for Similar Questions
Medium

Select two questions from the dataset that have similar meanings but different wording.

Instructions:
- Pass both questions through the trained model.
- Visualize their attention weights.
- Compare the words receiving the highest attention weights.

Question: Does the model focus on similar or different words when processing questions with similar meanings? What does this suggest about how attention works?

In [9]:
import matplotlib.pyplot as plt
import torch

similar_indices = [0, 1] 

plt.figure(figsize=(10, 4))
for i, idx in enumerate(similar_indices):
    question = df["question"][idx]
    words = question.split()
    
    with torch.no_grad():
        prediction, weights = model(X[idx:idx+1])
        
    attn_weights = weights[0, :len(words)].tolist()
    
    plt.subplot(1, 2, i+1)
    plt.bar(words, attn_weights, color='orange')
    plt.title(f"Similar Q{i+1}: {question}")
    plt.xticks(rotation=45)
    
    max_weight = max(attn_weights)
    max_word = words[attn_weights.index(max_weight)]
    print(f"Similar Q{i+1} ('{question}') -> Highest attention word: '{max_word}'")

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

<Figure size 1000x400 with 0 Axes>